# 01 · Ingestão para o S3 — Camada Bronze
**Tech Challenge Fase 3 · State of Data Brasil (Data Hackers/Bain)**

| Item | Descrição |
|---|---|
| **Objetivo** | Ingerir as 6 edições da pesquisa (2019–2025/26) na camada Bronze, particionadas por ano, de forma bruta e imutável. |
| **Origem** | CSVs Kaggle/Data Hackers — já versionados em `datalake/bronze/` neste repositório; `downloads/` local só é usado se for regenerar a partir de um novo download (upload manual no AWS) |
| **Destino** | `bronze/ano=YYYY/` (local: `datalake/` · AWS: `s3://<bucket>/datalake/`) |
| **Requisitos atendidos** | R1 (3 últimas pesquisas + histórico), R3 (ingestão e organização no S3) |
| **Segurança (AWS)** | Bucket privado (Block Public Access), SSE-S3, LabRole (menor privilégio) |

> **Escopo (decisão do Ricardo):** edições **2023, 2024 e 2025/26 = núcleo obrigatório**;
> edições **2019, 2021 e 2022 = contexto histórico** (séries longas), conforme regra 13.9.

> ⚠️ **Nota de reorganização (padrão Medallion).**
>
> O **código** das células abaixo já aponta para a estrutura atual do repositório:
> `../../datalake/{bronze,silver,gold}`, `../../consumption/charts` e as tabelas Gold
> renomeadas em inglês (`gold_roles`, `gold_salary_by_seniority`, …).
>
> **Atualização (26/08/2026):** as saídas abaixo já foram regravadas por reexecução real
> (PySpark 3.5.1 + JDK 17), com a nomenclatura atual — não são mais as saídas originais
> anteriores à reorganização.


In [1]:
# ============================================================
# PORTABILIDADE AWS GLUE (descomente APENAS no Glue Notebook)
# ============================================================
# %glue_version 4.0
# %worker_type G.1X
# %number_of_workers 2
# %idle_timeout 30
# No AWS, ajuste BASE para: s3://<seu-bucket>/datalake
# ============================================================

In [2]:
import os, shutil, csv

# ------------------------------------------------------------------
# Configuração de ambiente (local ↔ AWS)
# ------------------------------------------------------------------
IS_AWS = False                                   # True no AWS Academy Lab
BUCKET = "s3://stateofdata-fiap-tc3"             # ajustar ao seu bucket
BASE = f"{BUCKET}/datalake" if IS_AWS else "../../datalake"
BRONZE = f"{BASE}/bronze"

# Fontes brutas (CSVs baixados do Kaggle, fora do repositório) e volumetria oficial esperada.
# Os CSVs já estão versionados em datalake/bronze/ano=YYYY/ (ver README) — na execução normal
# a célula seguinte não copia nada (é idempotente). Ajuste os caminhos abaixo apenas se for
# regenerar a Bronze a partir de um novo download do Kaggle.
FONTES = {
    2019: "downloads/datahackerssurvey2019anonymousresponses.csv",
    2021: "downloads/State_of_Data_2021__Dataset__Pgina1.csv",
    2022: "downloads/State_of_data_2022.csv",
    2023: "downloads/State_of_data_BR_2023_Kaggle__df_survey_2023.csv",
    2024: "downloads/Final_Dataset__State_of_Data_2024__Kaggle__df_survey_2024.csv",
    2025: "downloads/Final_Dataset__State_of_Data_20252026__Kaggle.csv",
}
VOLUMETRIA_ESPERADA = {2019: 1765, 2021: 2645, 2022: 4271,
                       2023: 5293, 2024: 5217, 2025: 3495}

In [3]:
# ------------------------------------------------------------------
# Ingestão: cópia bruta e imutável para bronze/ano=YYYY/ (idempotente — não recopia se já existir)
# (no AWS: aws s3 cp <arquivo> s3://<bucket>/datalake/bronze/ano=YYYY/)
# ------------------------------------------------------------------
for ano, src in FONTES.items():
    destino = f"{BRONZE}/ano={ano}"
    alvo = f"{destino}/state_of_data_{ano}.csv"
    if os.path.exists(alvo):
        print(f"[ingestão] {ano} -> já existe em {alvo}, nada a fazer")
        continue
    os.makedirs(destino, exist_ok=True)                      # cria o prefixo da partição
    shutil.copy(src, alvo)                                   # cópia sem transformação (Bronze imutável)
    print(f"[ingestão] {ano} -> {alvo}")

[ingestão] 2019 -> já existe em ../../datalake/bronze/ano=2019/state_of_data_2019.csv, nada a fazer
[ingestão] 2021 -> já existe em ../../datalake/bronze/ano=2021/state_of_data_2021.csv, nada a fazer
[ingestão] 2022 -> já existe em ../../datalake/bronze/ano=2022/state_of_data_2022.csv, nada a fazer
[ingestão] 2023 -> já existe em ../../datalake/bronze/ano=2023/state_of_data_2023.csv, nada a fazer
[ingestão] 2024 -> já existe em ../../datalake/bronze/ano=2024/state_of_data_2024.csv, nada a fazer
[ingestão] 2025 -> já existe em ../../datalake/bronze/ano=2025/state_of_data_2025.csv, nada a fazer


In [4]:
# ------------------------------------------------------------------
# Validação: reconciliação de volumetria (integridade — pilar CIA)
# ------------------------------------------------------------------
print(f"{'ano':>5} | {'linhas':>7} | {'colunas':>7} | status")
for ano in sorted(FONTES):
    caminho = f"{BRONZE}/ano={ano}/state_of_data_{ano}.csv"
    with open(caminho, encoding="utf-8") as f:
        leitor = csv.reader(f)
        n_cols = len(next(leitor))          # cabeçalho
        n_rows = sum(1 for _ in leitor)     # registros (parser CSV trata quebras entre aspas)
    ok = "OK" if n_rows == VOLUMETRIA_ESPERADA[ano] else "DIVERGENTE"
    assert n_rows == VOLUMETRIA_ESPERADA[ano], f"{ano}: volumetria divergente!"
    print(f"{ano:>5} | {n_rows:>7} | {n_cols:>7} | {ok}")
print("\nBronze validada — pronta para o ETL (notebook 02).")

  ano |  linhas | colunas | status
 2019 |    1765 |     170 | OK
 2021 |    2645 |     356 | OK
 2022 |    4271 |     353 | OK
 2023 |    5293 |     399 | OK


 2024 |    5217 |     403 | OK
 2025 |    3495 |     388 | OK

Bronze validada — pronta para o ETL (notebook 02).
